# 17.3 模型进化 (Model Evolution)

模型进化关注模型如何随时间持续改进和适应新需求，是AI系统长期运维的核心。

本节涵盖：
- 模型升级与权重插值
- 能力扩展（新任务/新语言/新模态）
- 模型合并与模型路由
- 版本管理与回滚
- 产业级模型进化系统

## 1. 权重插值与模型合并

### 权重插值
在两个模型权重之间线性插值：$\theta_{interp} = \alpha \theta_1 + (1-\alpha) \theta_2$

### SLERP (Spherical Linear Interpolation)
在球面上插值，比线性插值更平滑：
$$\theta_{slerp} = \frac{\sin((1-\alpha)\Omega)}{\sin(\Omega)} \theta_1 + \frac{\sin(\alpha\Omega)}{\sin(\Omega)} \theta_2$$

### 模型合并方法
| 方法 | 原理 | 优势 | 适用场景 |
|------|------|------|----------|
| 线性合并 | 加权平均 | 简单 | 同架构模型 |
| SLERP | 球面插值 | 平滑 | 同架构模型 |
| Task Arithmetic | 任务向量加减 | 灵活 | 多任务合并 |
| TIES-Merging | 修剪+符号共识 | 减少干扰 | 多专家合并 |
| DARE | 随机丢弃+缩放 | 高效 | LoRA合并 |

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

def linear_merge(theta1, theta2, alpha=0.5):
    return alpha * theta1 + (1 - alpha) * theta2

def slerp(theta1, theta2, alpha=0.5, eps=1e-8):
    flat1 = theta1.flatten()
    flat2 = theta2.flatten()
    dot = F.cosine_similarity(flat1.unsqueeze(0), flat2.unsqueeze(0)).clamp(-1, 1)
    omega = torch.acos(dot)
    if omega.abs() < eps:
        return linear_merge(theta1, theta2, alpha)
    sin_omega = torch.sin(omega)
    result = (torch.sin((1 - alpha) * omega) / sin_omega) * theta1 + (torch.sin(alpha * omega) / sin_omega) * theta2
    return result

def task_arithmetic(base_theta, task_vectors, coefficients=None):
    if coefficients is None:
        coefficients = [1.0] * len(task_vectors)
    merged = base_theta.clone()
    for tv, coef in zip(task_vectors, coefficients):
        merged += coef * tv
    return merged

def ties_merging(task_vectors, base_theta, k=0.2):
    trimmed = []
    for tv in task_vectors:
        threshold = torch.quantile(tv.abs().flatten(), 1 - k)
        trimmed_tv = tv * (tv.abs() >= threshold).float()
        trimmed.append(trimmed_tv)
    signs = []
    for tv in trimmed:
        signs.append(torch.sign(tv))
    sign_sum = sum(signs)
    consensus_sign = torch.sign(sign_sum)
    merged_delta = torch.zeros_like(base_theta)
    for tv in trimmed:
        mask = (torch.sign(tv) == consensus_sign).float()
        merged_delta += tv * mask
    n_agree = (sign_sum.abs() > 0).float().sum().clamp(min=1)
    merged_delta = merged_delta / n_agree
    return base_theta + merged_delta

base = torch.randn(64, 64) * 0.1
task1 = torch.randn(64, 64) * 0.05
task2 = torch.randn(64, 64) * 0.05
model1 = base + task1
model2 = base + task2

print('=== Model Merging Methods ===')
for alpha in [0.0, 0.25, 0.5, 0.75, 1.0]:
    merged_linear = linear_merge(model1, model2, alpha)
    merged_slerp_val = slerp(model1, model2, alpha)
    diff = (merged_linear - merged_slerp_val).norm()
    print(f'alpha={alpha:.2f}: linear_norm={merged_linear.norm():.3f}, slerp_norm={merged_slerp_val.norm():.3f}, diff={diff:.4f}')

merged_ta = task_arithmetic(base, [task1, task2], [1.0, 1.0])
merged_ties = ties_merging([task1, task2], base, k=0.2)
print(f'\nTask Arithmetic: norm={merged_ta.norm():.3f}')
print(f'TIES Merging: norm={merged_ties.norm():.3f}')
print(f'\nKey: SLERP is smoother than linear merge for weight interpolation.')
print(f'Task Arithmetic adds/subtracts task vectors from a base model.')
print(f'TIES trims noisy parameters and resolves sign conflicts.')

## 2. 能力扩展

### 新任务扩展
在已有模型基础上添加新任务能力：
- **扩展分类头**：增加输出类别
- **添加适配器**：新任务用新适配器，旧任务不受影响
- **渐进式训练**：先冻结旧部分，只训练新部分

### 新语言扩展
扩展模型到新语言：
- **词表扩展**：添加新语言token
- **嵌入初始化**：用相近语言嵌入初始化
- **两阶段训练**：先学新语言基础，再学新语言能力

### 新模态扩展
扩展模型到新模态（视觉/音频）：
- **模态编码器**：添加新模态的编码器
- **模态投影**：将新模态映射到语言空间
- **多模态对齐**：对比学习对齐不同模态

In [ ]:
class EvolvableModel(nn.Module):
    def __init__(self, d=64, n_classes=10, n_layers=2):
        super().__init__()
        self.d = d
        self.layers = nn.ModuleList([nn.Linear(d, d) for _ in range(n_layers)])
        self.head = nn.Linear(d, n_classes)
        self.adapters = nn.ModuleDict()

    def add_layer(self):
        self.layers.append(nn.Linear(self.d, self.d))

    def expand_head(self, new_classes=5):
        old_head = self.head
        old_classes = old_head.out_features
        new_head = nn.Linear(self.d, old_classes + new_classes)
        with torch.no_grad():
            new_head.weight[:old_classes] = old_head.weight
            new_head.bias[:old_classes] = old_head.bias
        self.head = new_head

    def add_adapter(self, task_id, rank=4):
        adapter = nn.ModuleDict({
            'down': nn.Linear(self.d, rank, bias=False),
            'up': nn.Linear(rank, self.d, bias=False)
        })
        nn.init.zeros_(adapter['up'].weight)
        self.adapters[str(task_id)] = adapter

    def forward(self, x, task_id=None):
        for layer in self.layers:
            x = F.relu(layer(x))
        if task_id is not None and str(task_id) in self.adapters:
            adapter = self.adapters[str(task_id)]
            x = x + adapter['up'](adapter['down'](x))
        return self.head(x)

class ModelVersionManager:
    def __init__(self):
        self.versions = {}
        self.current_version = 0

    def save_version(self, model, tag=''):
        self.current_version += 1
        self.versions[self.current_version] = {
            'state_dict': {n: p.clone() for n, p in model.state_dict().items()},
            'tag': tag,
            'n_params': sum(p.numel() for p in model.parameters())
        }
        return self.current_version

    def restore_version(self, model, version_id):
        if version_id in self.versions:
            model.load_state_dict(self.versions[version_id]['state_dict'])
            return True
        return False

    def list_versions(self):
        return {v: {'tag': d['tag'], 'n_params': d['n_params']} for v, d in self.versions.items()}

model = EvolvableModel(d=64, n_classes=10)
vm = ModelVersionManager()

v1 = vm.save_version(model, 'initial_10class')
model.add_adapter('task_b', rank=4)
v2 = vm.save_version(model, 'added_adapter')
model.expand_head(new_classes=5)
v3 = vm.save_version(model, 'expanded_15class')
model.add_layer()
v4 = vm.save_version(model, 'deeper_model')

print('=== Model Evolution ===')
print(f'Version history:')
for v, info in vm.list_versions().items():
    print(f'  v{v}: tag={info["tag"]}, params={info["n_params"]:,}')

x = torch.randn(4, 64)
out = model(x)
print(f'\nCurrent model output shape: {out.shape}')

vm.restore_version(model, v2)
out_restored = model(x)
print(f'After restoring v2: output shape: {out_restored.shape}')
print(f'\nKey: Model versioning enables rollback to any previous state.')
print(f'Capability expansion (new classes, adapters, layers) preserves old functionality.')

## 3. 模型路由与混合专家

### 模型路由
根据输入动态选择最合适的模型版本：
- **语言路由**：不同语言用不同模型
- **任务路由**：不同任务用不同专家
- **难度路由**：简单查询用小模型，复杂查询用大模型

### 产业实践
- **RouteLLM**：训练路由器在大小模型间选择，节省成本
- **MoE**：混合专家，每个token只激活部分参数
- **模型级联**：简单→复杂逐级尝试

### 模型进化总结
| 策略 | 方法 | 成本 | 风险 |
|------|------|------|------|
| 权重插值 | 线性/SLERP | 低 | 中 |
| 任务算术 | 向量加减 | 低 | 低 |
| 适配器扩展 | 新任务新适配器 | 中 | 低 |
| 模型路由 | 动态选择 | 低 | 低 |
| 从头训练 | 完全重训 | 高 | 低 |

**前沿方向**：
- **Model Stock**：基于几何的模型合并
- **Git-like模型管理**：分支、合并、回滚
- **自动架构搜索**：NAS驱动的模型进化

In [ ]:
class ModelRouter(nn.Module):
    def __init__(self, d=64, n_models=3):
        super().__init__()
        self.n_models = n_models
        self.router = nn.Sequential(
            nn.Linear(d, 32), nn.ReLU(), nn.Linear(32, n_models), nn.Softmax(dim=-1)
        )
        self.difficulty_estimator = nn.Sequential(
            nn.Linear(d, 16), nn.ReLU(), nn.Linear(16, 1), nn.Sigmoid()
        )

    def forward(self, x):
        routing_probs = self.router(x.mean(dim=0, keepdim=True) if x.dim() > 1 else x)
        difficulty = self.difficulty_estimator(x.mean(dim=0, keepdim=True) if x.dim() > 1 else x)
        return routing_probs, difficulty

class CascadedModels:
    def __init__(self, models, router, confidence_thresholds=None):
        self.models = models
        self.router = router
        self.thresholds = confidence_thresholds or [0.7, 0.5]
        self.stats = {'small': 0, 'medium': 0, 'large': 0}

    def forward(self, x):
        routing_probs, difficulty = self.router(x)
        if difficulty.item() < self.thresholds[0]:
            self.stats['small'] += 1
            return self.models[0](x), 0
        elif difficulty.item() < self.thresholds[1]:
            self.stats['medium'] += 1
            return self.models[1](x), 1
        else:
            self.stats['large'] += 1
            return self.models[2](x), 2

small = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 10))
medium = nn.Sequential(nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 10))
large = nn.Sequential(nn.Linear(64, 128), nn.ReLU(), nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 10))
router = ModelRouter(d=64, n_models=3)

cascade = CascadedModels([small, medium, large], router)

print('=== Model Routing & Cascade ===')
for i in range(10):
    x = torch.randn(1, 64) * (1 + i * 0.3)
    out, model_idx = cascade(x)
    probs, diff = router(x)
    print(f'  Input {i+1}: difficulty={diff.item():.3f}, selected=model_{model_idx} ({["small","medium","large"][model_idx]})')

print(f'\nRouting stats: {cascade.stats}')
print(f'\nKey: Model routing saves compute by using smaller models for easy inputs.')
print(f'Cascade tries small model first, escalates to larger if confidence is low.')
print(f'In production, this can reduce serving cost by 50-70% for typical workloads.')